# Name
Please write your name down here:

YOUR ANSWER HERE

**Association tests between variants and continuous phenotypes**
Yesterday we had learned how to test for association between categorical variables: our variables were genotype and color.

The phenotype data provided in the multi-omics publication is not categorical. We thus have to adapt our approach to deal with continuous data. In the lecture you saw that linear (Pearson's), logistic, or rank-based (Mann-Whitney U) tests can be used when one of the variables is categorical, and the other is categorical.

If the categorical variable is binary, and the continuous variable is normally distributed, a t-test can be used: Student's t-test if the variances of the two groups is identical, and the Welch's t-test otherwise.

In this notebook we will only use Pearson's correlation and the Mann-Whitney U-test.

(And a little note on nomenclature: the conventional name of Pearson's correlation is "point-biserial correlation" if one of the variables is binary. They are exactly the same thing, but if you ever come across the latter, remember that it refers to a special case of Pearson's correlation. Despite our genotypes being *almost* binary, we will keep using the term Pearson's correlation.)

In [ ]:
import math
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.sandbox.stats.multicomp import multipletests

# Preparing the data
## Loading the genotype data
Loading the genotype data is routine by now: separating data from metadata, and converting the B-D-H-U labels into numbers. This time, we will avoid wasting any of our precious data, and instead of dropping strains with too many unknowns, we will work our way around them. We will treat the unknown `U` genotypes as `NaN` values.

In [ ]:
genotype = pd.read_csv("https://github.com/Practical-Integrative-Bioinformatics/00_Introduction/blob/main/data/genotype.txt?raw=True", sep="\t", comment="@")
genotype.set_index("Locus", inplace=True)
geno_meta = genotype.iloc[:, :3]
geno_bdh = genotype.iloc[:, 3:]
geno = geno_bdh.replace(['B', 'H', 'D', 'U'], [0, 1, 2, np.nan])
geno.columns.name = 'strain'

geno['C57BL/6J'] = 0
geno['DBA/2J'] = 2

## Loading the phenotype tables

As for the phenotype spreadsheets, we will always use the `split_cd_hfd(...)`-transformed version from now on. We have provided its code so you don't have to dig it up from Tuesday. Let's convert all sheets with `split_cd_hfd` and forget about it. From now on, the `phenotypes` dictionary will always contain the nice CD/HFD-indexed DataFrames.

In [ ]:
phenotypes_ugly = pd.read_excel('https://github.com/Practical-Integrative-Bioinformatics/00_Introduction/blob/main/data/phenotype.xlsx?raw=True', sheet_name=None, na_values='x', index_col=0)

# The VO2Max table doesn't follow their own column naming conventions, so we fix that first
def rename_vo2_cols(colname):
    return colname.replace('CD', 'CD_').replace('HFD', 'HFD_').replace('__', '_')

phenotypes_ugly['VO2Max'].rename(columns=rename_vo2_cols, inplace=True)

def split_cd_hfd(input_df):
    input_df.index.name = 'strain'  # change that weird @format=column Excel index label to something meaningful
    input_cd = input_df.filter(regex=r'_CD|CD_') # loc[:, (input_df.columns.str.contains('CD_')) | (input_df.columns.str.contains('_CD'))]
    input_hfd = input_df.filter(regex=r'_HFD|HFD_') #loc[:, (input_df.columns.str.contains('HFD_')) | (input_df.columns.str.contains('_HFD'))]

    input_cd.columns = input_cd.columns.str.replace(r'_CD|CD_', '', regex=True)
    input_hfd.columns = input_hfd.columns.str.replace(r'_HFD|HFD_', '', regex=True)
    
    input_cd.insert(0, 'diet', 'CD')
    input_hfd.insert(0, 'diet', 'HFD')
    
    kept_columns = input_cd.columns.intersection(input_hfd.columns)
    # kept_columns = input_cd.columns & input_hfd.columns  # this also works
    
    df_both = pd.concat([input_cd, input_hfd], sort=False)[kept_columns]
    df_both.columns.name = 'experiment'  # added only later
    
    return df_both.set_index('diet', append=True).sort_index()

# for every sheet (other than the last empty one) we do a split_cd_hfd and then drop
# columns that are admittedly problematic for statistical testing.
# Those columns have "KNOWN_BATCH_EFFECT_BY_COHORT_ORDER" in their names, which we can
# filter out with a tricky regex (sadly .filter() doesn't have an invert=True keyword)

phenotypes = {sheet_name: split_cd_hfd(sheet).filter(regex=r'^(?!.*BATCH).*$')
              for sheet_name, sheet in phenotypes_ugly.items() if sheet_name != 'NEW'}

phenotypes['Biochemistry']  # nice CD/HFD DataFrame without the bad columns

## Reindexing the genotype DataFrame to match the format of the phenotype DataFrames

Since we will spend the day comparing genotypes with phenotypes, it is worth preprocessing our data a bit to make comparisons easier. The `geno` DataFrame has strains as columns, whereas the `phenotypes[...]` DataFrames have strains as rows. Furthermore, due to most phenotype measurements having been performed twice (once under the CD and once under the HFD diet) we have two rows per each strain in the phenotype DataFrames.

Reshaping DataFrames is a common task, and `pandas` can do the work for us with the `.reindex()` method.

Let's reindex the `geno` DataFrame to match the index structure of the phenotype DataFrames. We will have to do two things: first transpose `geno` and then `.reindex(...)` it with the index object of any phenotype DataFrame from the `phenotypes` dictionary. Since the phenotype DataFrames have a MultiIndex, we will also have to tell the `reindex` method which level we want to match it on. Store the re-indexed genotype DataFrame in `geno2`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert 'geno2' in locals()

<div class="alert alert-block alert-info"><b>Note: </b> Since the genotype and phenotype DataFrames contained different sets of strains, <code>pandas</code> had to discard 38 strains from <code>geno</code> and introduce 2 strains with all missing values. This was the price of getting the genotype and phenotype DataFrames lined up.</div>

Of course we couldn't have used those 38 discarded strains for anything, since we don't have any phenotype information for them. The 2 extra strains with fully unknown genotypes are equally useless, but they don't bother us: while we could have removed them from all 11 phenotype sheets instead of inserting them into the `geno2` DataFrame, it's not worth the effort, since we already have to deal with a lot of missing data anyway. Remember, we have other `NaN`s in both the genotype and phenotype tables.

# Implement Pearson's r-based test for correlation between a genotype and a phenotype

Choose a phenotype whose genetic associations you want to study, for example `Glucose_[mmol/L]` from the `Biochemistry` sheet. Store this Series under the name `one_pheno`.

Likewise, choose a locus from `geno2`, e.g. `rs3677240`. Store its genotypes in the Series `one_geno`.

Perform a Pearson's correlation test between the two using `stats.pearsonr`. The function expects two arrays or Series with matching elements. Thankfully our previous reindexing has already taken care of that.

Unfortunately, as you'll immediately see, the `stats.pearsonr` function can't handle `NaN` values. So make sure to remove elements from *both* Series where *either of them* is an NaN.

You can achieve this with boolean slicing: a Series' `.isna()` method gives you a boolean vector with `True` elements at the missing values' positions. You can combine and invert boolean vectors with logical operators, and use the resulting mask to slice both the genotype and phenotype Series at the necessary positions.

If everything works, wrap these two steps (i.e. NaN masking and Pearson correlation) into a function named `pearson_between(one_geno, one_pheno)` which takes the two Series as inputs, and returns the Pearson correlation test's p-value.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert 'one_pheno' in locals()
assert 'one_geno' in locals()
assert math.isclose(pearson_between(geno2['rs3677240'], phenotypes['Biochemistry']['Glucose_[mmol/L]']), 0.0093, abs_tol=1e-4)

# Visualize the phenotype value distributions of the B and D genotype

Two transparent histograms on the same plot should do the job. **Based on the p-value that you had previously calculated, does the histogram match your expectations?**

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

# Perform a correlation test between *all* loci and one phenotype

Now that you have the `pearson_between` function to perform the correlation test between a genotype and a phenotype Series, you might as well `.apply` it to the enitre `geno2` DataFrame and calculate p-values for all loci.

Remember, the `apply` method acts on the DataFrame's columns by default, which is exactly what we need. It can also pass on extra keyword arguments to the function, so use the `phenotypes['Biochemistry']['Glucose_[mmol/L]']` to correlate the different genotypes with.

Save the result under the name `one_pheno_vs_all_geno`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert 'one_pheno_vs_all_geno' in locals()

## Find the locus with the lowest p-value, and visualize the phenotype distribution for B and D genotypes

Create the same histogram again, except this time use the locus with the most significant PPearson's correlation to the phenotype (i.e. with the smallest p-value). You can either find the name of the locus by sorting and looking at the top, or using `.idxmin()`.

**How is the separation on this plot compared to the last one?**

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

## Create a Manhattan plot for the `Glucose_[mmol/L]` phenotype

We see that there are a good bunch of SNPs that appear strongly associated with glucose levels in mice. **Draw a Manhattan plot to see where they cluster on the genome, and do a bit of research to confirm whether what you found is confirmed by the literature on the topic.**

You should obviously re-use your code from the previous notebook to create the plot. You can (and of course should) perform a Benjamini-Hochberg correction on your p-values, but seeing the magnitude of the p-values, the best ones will clearly pass, and since we aren't planning to publish our findings, we won't complain if you just plot them as-is.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

# Perform a correlation test between all loci and all phenotypes of a sheet

The inevitable next step: running the Pearson correlation test between all pairs of loci and phenotypes.

If you can run the test between a single phenotype and all columns of `geno2`, you can also run it for every phenotype in the Biochemistry sheet: you just have to iterate over `phenotypes['Biochemistry']`'s columns using the DataFrame's `.iteritems()` method.

All you have to do is collect the resulting p-value Series in a list, concatenate them with `pd.concat` along the horizontal (`1`) axis, and name the columns. You can either name the columns after concatenation, but you can also think ahead and take care of it inside the iteration, right after each p-value Series has just been created. (If `pd.concat` receives a list of named Series, it will know how to name the columns of the concatenated DataFrame).

Name the resulting DataFrame `all_pheno_vs_all_geno`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert 'all_pheno_vs_all_geno' in locals()

# Create a Mann-Whitney U test function similar to `pearson_between`

The Mann-Whitney U test compares a strictly binary categorical variable with a continuous variable. Or in other words: it compares two groups of contiuous values.

Create a `mwu_between(one_geno, one_pheno)` function which performs the Mann-Whitney U test instead of Pearson's correlation. But watch out! The expected input of `stats.mannwhitneyu` is different from that of `stats.pearsonr`: instead of passing a genotype Series and a phenotype Series, you will have to pass two phenotype Series, one containing the phenotype values of B mice (`one_geno == 0`), and the other containing the phenotype values of D mice (`one_geno == 2`).

While having to slice `one_pheno` may seem like extra work, it has an upside: you don't have to bother with the parallel masking of `NaN` values, because `.dropna()` on both Series independently will do just fine.

Try out your `mwu_between` function on the same locus and phenotype that you had used when testing `pearson_between`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert math.isclose(mwu_between(geno2['rs3677240'], phenotypes['Biochemistry']['Glucose_[mmol/L]']), 0.0173, abs_tol=1e-4)

# Calculate Mann-Whitney p-values for all loci-phenotype combinations for the Biochemistry sheet

This should be a copy-paste job. It will give you another big matrix with a lot of p-values in them. Let's call it `all_pheno_vs_all_geno_mwu`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert 'all_pheno_vs_all_geno_mwu' in locals()

To make use of this data, create a `-log10` scatter-plot comparing all of its values with the Pearson's p-values in `all_pheno_vs_all_geno`. **What is the difference between the p-values of both tests?**

<div class="alert alert-block alert-info"><b>Hint 1: </b> You can turn a numpy matrix into a 1-dimensional array with the <code>.flatten()</code> method. You can access a DataFrame's underlying numpy matrix with <code>.values</code>.</div>

<div class="alert alert-block alert-info"><b>Hint 2: </b> To "zoom in" on the interesting p-values, you can change your plot type to log-log (or equivalently, plot the log10 of the p-values).</div>

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

# Multiple testing correction

This is a tricky one. Having calculated p-values for every locus-phenotype pair, we now have a big matrix with tens of thousands of p-values. That's a lot of tests, and therefore they deserve a big, strict correction.

As you will find, the function `multipletests` can only deal with a 1-dimensional array of p-values. This means we will have to:
1. access the DataFrame's underlying numpy matrix with `.values`
2. flatten it into a 1d array
3. run `multipletests(..., method='fdr_bh')` on it
4. reshape it back to the original dimensions
5. turn it back to a DataFrame with the proper indices and columns.
6. Store it as `all_pheno_vs_all_geno_pearson_bh`

These steps will take four lines of code.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert 'all_pheno_vs_all_geno_pearson_bh' in locals()

## Are there any locus-phenotype associations that survive multiple testing correction?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

## And the question again: could we have overcorrected the data? Why?

YOUR ANSWER HERE

# What confounding factors may have influenced our analysis? How could they be dealt with?

YOUR ANSWER HERE

# Optional: Violin plot for a phenotype sheet

This is a hard one for those that have extra time on their hands.

Pick a phenotype sheet, and create a split violin for every phenotype, stratified by the genotypes on the locus that has the strongest association with the phenotype. A lot of techniques come together in this task, but I guarantee it will fill you with a sense of accomplishment if you pull it off. You have all the ingredients, but it's not trivial to put them together.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()